# Day 33 / 100 — FruitCount V4 | SAVE VERSION FIRST 🍅

**Main goal:** obtain the trained checkpoint, real validation metrics, and a tested download archive without running long optional per-image counting, extra `model.val`, or test inference in the same Kaggle Save Version.

**Based on:** the uploaded original 24-cell V3 notebook. Same dataset, official splits (477/105/95), six tomato ripeness classes, and pretrained YOLO11n. **No raw images are included in exported ZIP.**

**Two-phase policy:** Phase 1 (this notebook) trains and packages. Phase 2 (separate notebook) loads the saved checkpoint and performs extra counting/test evaluations. Prior V2/V3 test exposure means the 95-image test is *development-aware*, not blind. A Kaggle run may still fail due to resource/network limits; this notebook avoids unbounded post-training inference and packages directly after training.

## 0 — Environment and resource-safe configuration

- GPU required, Internet On or add local trusted `yolo11n.pt` to inputs.
- `TRAIN_MODE='train'` by default; change to `'reuse'` only if a previously exported checkpoint is mounted under `RESTORE_CHECKPOINT`.
- Check the Kaggle output tab for `FruitCount_Day33_V4_Training_Results.zip` after the version finishes.

In [1]:
import os, sys, gc, json, csv, shutil, hashlib, random, time, platform, warnings
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageOps
import torch

try:
    import ultralytics
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.3,<9'])
    import ultralytics
from ultralytics import YOLO

DATA_ROOT=Path('/kaggle/input/datasets/wafabila/fruitcount-dataset')
WORK=Path('/kaggle/working/fruitcount_day33_v4')
REPORTS=WORK/'reports'; PLOTS=WORK/'assets'/'plots'; EXAMPLES=WORK/'assets'/'examples'
MODELS=WORK/'models'; YOLO_DATA=WORK/'yolo_dataset'; RUNS=WORK/'train_runs'
for d in (REPORTS,PLOTS,EXAMPLES,MODELS,YOLO_DATA,RUNS):d.mkdir(parents=True,exist_ok=True)

SEED=42
CLASSES=['Green','Breakers','Turning','Pink','Light Red','Red']
IMG_SIZE=960
BATCH_SIZE=4
EPOCHS=60
PATIENCE=12
WORKERS=0
DEVICE=0 if torch.cuda.is_available() else 'cpu'
PRETRAINED_WEIGHTS='yolo11n.pt'
RUN_FULL_SHA256=False # source hashes already audited in V3; optional slower independent check
TRAIN_MODE='train'  # 'train' or 'reuse' (skip training with a verified checkpoint)
RESTORE_CHECKPOINT=None # example '/kaggle/input/my-fruitcount-model/best.pt'
RUN_NAME='yolo11n_v4'

assert DATA_ROOT.is_dir(),f'Dataset missing: {DATA_ROOT}'
assert torch.cuda.is_available(), 'Enable Kaggle GPU before Save Version'
assert TRAIN_MODE in {'train','reuse'}
assert IMG_SIZE%32==0 and len(CLASSES)==6
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
print('Python',sys.version.split()[0],'Torch',torch.__version__,'Ultralytics',ultralytics.__version__)
print('GPU',torch.cuda.get_device_name(0),'Input mounted:',DATA_ROOT.exists())
config={'seed':SEED,'image_size':IMG_SIZE,'batch_size':BATCH_SIZE,'epochs_max':EPOCHS,
        'patience':PATIENCE,'workers':WORKERS,'architecture':'YOLO11n pretrained',
        'ultralytics':ultralytics.__version__,'torch':torch.__version__,
        'phase':'training_and_export_only','run_full_sha256':RUN_FULL_SHA256,
        'train_mode':TRAIN_MODE}
(REPORTS/'experiment_config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Python 3.13.16 Torch 2.11.0+cu130 Ultralytics 8.4.203
GPU Tesla T4 Input mounted: True


297

## 1 — Source identification and provenance

In [2]:
def find_one(relative):
    direct=DATA_ROOT/relative
    if direct.is_file():return direct
    matches=[p for p in DATA_ROOT.rglob(Path(relative).name) if p.as_posix().endswith(relative)]
    if len(matches)!=1:raise FileNotFoundError(f'Expected one {relative}; got {len(matches)}')
    return matches[0]

COCO_PATH={s:find_one(f'dataset/detection/coco/instances_{s}.json') for s in ('train','val','test')}
SPLIT_PATH={s:find_one(f'splits/images_{s}.csv') for s in ('train','val','test')}
MANIFEST_PATH=find_one('metadata/image_manifest.csv')
SOURCE_README=find_one('README.md');SOURCE_LICENSE=find_one('LICENSE')
source_files=[p for p in DATA_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png'} and 'images' in p.parts]
by_name=defaultdict(list)
for p in source_files:by_name[p.name].append(p)
assert all(len(v)==1 for v in by_name.values()),'Duplicate image basename: explicit mapping required'
IMAGE_PATHS={name:files[0] for name,files in by_name.items()}
COCO={s:json.loads(p.read_text(encoding='utf-8')) for s,p in COCO_PATH.items()}
SPLIT_CSV={s:pd.read_csv(p) for s,p in SPLIT_PATH.items()}
MANIFEST=pd.read_csv(MANIFEST_PATH)
print('Source:',SOURCE_README.read_text(encoding='utf-8',errors='replace')[:1100])
print('License:',SOURCE_LICENSE.read_text(encoding='utf-8',errors='replace')[:300])
print('Images discovered:',len(IMAGE_PATHS))
(REPORTS/'source_provenance.json').write_text(json.dumps({
 'source_dataset':'AerialYield-T2M','kaggle_input':str(DATA_ROOT),'dataset_link':'https://www.kaggle.com/datasets/wafabila/fruitcount-dataset',
 'license_excerpt':SOURCE_LICENSE.read_text(encoding='utf-8',errors='replace')[:1200],
 'readme_excerpt':SOURCE_README.read_text(encoding='utf-8',errors='replace')[:2200],
 'access_time_utc':datetime.now(timezone.utc).isoformat(),
 'limitations':['Source license and redistribution rules must be verified','Close-range and drone imagery may yield domain shift']
},ensure_ascii=False,indent=2),encoding='utf-8')

Source: # AerialYield-T2M

A dual-modality greenhouse tomato dataset with six-stage ripeness masks, instance annotations
and fruit counts. Full methodology, technical validation, and reference baselines are described
in the accompanying Data Descriptor manuscript.

## Contents

| Folder / file | Contents | Notes |
|---|---|---|
| `dataset_images.zip` | 677 RGB JPEG images from both acquisition sources (535 close-range, 142 drone). | |
| `dataset_masks_binary.zip` | Lossless binary PNG masks, one subfolder per ripeness class: `Green/`, `Breakers/`, `Turning/`, `Pink/`, `Light Red/`, `Red/`. | 4,739 mask files. |
| `dataset_masks_semantic.zip` | Single-channel semantic maps, values 0 (background) to 6 (ripeness classes 1-6 in Table order). No 255 ignore value is used. | 677 files. |
| `dataset_masks_overall.zip` | Overall-fruit masks, independent of ripeness class. | |
| `metadata.zip` | `image_manifest.csv` (image identifier, source, modality, split, geometry, raw per-class counts, the 

3699

## 2 — Official split and bounding-box audit

The source's original images and COCO annotations are checked before training. Exact SHA-256 is optional to avoid repeated ~3 GB reads; disabling hashing must not be described as completing the cross-split hash test in this run.

In [3]:
EXPECTED={'train':477,'val':105,'test':95}
EXPECTED_BOX={'train':10050,'val':1938,'test':2202}
issues=[];warnings_list=[];records={};boxes_by_name={};class_counts={};all_names=set();annotation_ids=Counter()
for split,data in COCO.items():
    catmap={int(c['id']):c['name'] for c in data['categories']}
    if catmap!={i+1:name for i,name in enumerate(CLASSES)}:issues.append(f'{split}: unexpected class mapping {catmap}')
    names=[Path(str(x['file_name']).replace('\\','/')).name for x in data['images']]
    ids=[x['id'] for x in data['images']]
    if len(set(names))!=len(names) or len(set(ids))!=len(ids):issues.append(f'{split}: duplicate names or image ids')
    if all_names.intersection(names):issues.append(f'{split}: overlapping image names')
    all_names.update(names)
    if set(SPLIT_CSV[split]['filename'].astype(str))!=set(names):issues.append(f'{split}: split CSV/COCO disagreement')
    if len(names)!=EXPECTED[split] or len(data['annotations'])!=EXPECTED_BOX[split]:issues.append(f'{split}: count differs from audited source')
    id_to_image={x['id']:x for x in data['images']}
    boxes=defaultdict(list);class_counts[split]=Counter()
    local_annotation_ids=set()
    for a in data['annotations']:
        aid=a['id'];annotation_ids[aid]+=1
        if aid in local_annotation_ids:issues.append(f'{split}: duplicate annotation id {aid}')
        local_annotation_ids.add(aid)
        if a['image_id'] not in id_to_image:issues.append(f'{split}: orphan annotation {aid}');continue
        im=id_to_image[a['image_id']];class_id=int(a['category_id'])-1
        if class_id not in range(6):issues.append(f'{split}: bad category {class_id}');continue
        xywh=np.asarray(a['bbox'],dtype=np.float64)
        x,y,w,h=xywh.tolist()
        if not np.isfinite(xywh).all() or w<=0 or h<=0 or x<-1e-3 or y<-1e-3 or x+w>im['width']+.001 or y+h>im['height']+.001:
            issues.append(f'{split}: unsafe box annotation {aid}');continue
        name=Path(str(im['file_name']).replace('\\','/')).name
        boxes[name].append([x,y,x+w,y+h,class_id]);class_counts[split][CLASSES[class_id]]+=1
    for im in data['images']:
        name=Path(str(im['file_name']).replace('\\','/')).name
        records[name]={'split':split,'width':int(im['width']),'height':int(im['height'])}
        boxes_by_name[name]=np.asarray(boxes[name],dtype=np.float32).reshape(-1,5)
if len(all_names)!=677 or set(IMAGE_PATHS)!=all_names:issues.append('677-image COCO/source file inventory mismatch')
if MANIFEST['filename'].duplicated().any():issues.append('Manifest duplicate filenames')
if set(MANIFEST['filename'].astype(str))!=all_names:issues.append('Manifest file names mismatch')
if 'split' in MANIFEST:
    manifest_split=dict(zip(MANIFEST.filename.astype(str),MANIFEST.split.astype(str)))
    if any(manifest_split.get(n)!=v['split'] for n,v in records.items()):issues.append('Manifest split mismatch')
counts_num=pd.to_numeric(MANIFEST['Total'],errors='coerce')
if counts_num.isna().any() or (counts_num<0).any() or not np.allclose(counts_num,counts_num.round()):issues.append('Invalid raw Total label')
assert not issues,'Audit failed: '+repr(issues[:12])
GT_RAW_COUNT=dict(zip(MANIFEST.filename.astype(str),counts_num.astype(int)))
assert sum(GT_RAW_COUNT.values())==14303
assert sum(len(v) for v in boxes_by_name.values())==14190
print('AUDIT: split images',EXPECTED,'filtered instances',EXPECTED_BOX,'raw fruit',sum(GT_RAW_COUNT.values()))
print('COCO annotation IDs repeated across splits:',sum(v>1 for v in annotation_ids.values()))
# Decode / dimensions; small number of images, do not retain raw decoded tensors in memory
bad_decodes=[];bad_dimensions=[];sha_groups=defaultdict(list)
for i,(name,p) in enumerate(sorted(IMAGE_PATHS.items()),1):
    try:
        with Image.open(p) as im:
            im.load()
            if im.size!=(records[name]['width'],records[name]['height']):bad_dimensions.append(name)
    except Exception as exc:bad_decodes.append((name,str(exc)[:100]))
    if RUN_FULL_SHA256:
        h=hashlib.sha256()
        with p.open('rb') as f:
            for chunk in iter(lambda:f.read(4*1024*1024),b''):h.update(chunk)
        sha_groups[h.hexdigest()].append(name)
    if i%150==0:print(f'Audited {i}/{len(IMAGE_PATHS)}')
exact_dupes=[ns for ns in sha_groups.values() if len(ns)>1]
cross_dupes=[ns for ns in exact_dupes if len({records[n]['split'] for n in ns})>1]
if bad_decodes or bad_dimensions or cross_dupes:raise RuntimeError(f'Unsafe data: decodes={bad_decodes[:3]} dimensions={bad_dimensions[:3]} cross-split hashes={cross_dupes[:3]}')
# Filename-block heuristic: same source prefix / adjacent 300-index frames, imperfect grouping
blocks=defaultdict(set);unparsed=[]
for name,rec in records.items():
    try:
        prefix,index=Path(name).stem.rsplit('_',1)
        blocks[(prefix,int(index)//300)].add(rec['split'])
    except (ValueError,IndexError):unparsed.append(name)
shared_blocks=[str(k) for k,v in blocks.items() if len(v)>1]
if shared_blocks:warnings_list.append(f'Potential cross-split acquisition blocks: {len(shared_blocks)}')
if unparsed:warnings_list.append(f'Unparsed filename block groups: {len(unparsed)}')
summary={'images':{s:sum(r['split']==s for r in records.values()) for s in EXPECTED},
         'boxes':{s:sum(len(boxes_by_name[n]) for n,r in records.items() if r['split']==s) for s in EXPECTED},
         'raw_visible_fruit':int(sum(GT_RAW_COUNT.values())), 'coco_boxes':14190,
         'sha256_audit_ran':RUN_FULL_SHA256,'exact_duplicates':len(exact_dupes) if RUN_FULL_SHA256 else None,'exact_cross_split':len(cross_dupes) if RUN_FULL_SHA256 else None,
         'inferred_block_groups':len(blocks),'shared_blocks':shared_blocks,'unparsed_block_names':len(unparsed),
         'bad_decodes':bad_decodes,'bad_dimensions':bad_dimensions,'warnings':warnings_list,
         'limitations':['No perceptual near-duplicate or plant-identity audit','Filename-block heuristic not verified as plant-independent','Previously seen test split (V2) is not blind in V3']}
(REPORTS/'dataset_audit.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('[PASS] Dataset audit',json.dumps({k:summary[k] for k in ('images','boxes','exact_cross_split','shared_blocks')}))

AUDIT: split images {'train': 477, 'val': 105, 'test': 95} filtered instances {'train': 10050, 'val': 1938, 'test': 2202} raw fruit 14303
COCO annotation IDs repeated across splits: 2202
Audited 150/677
Audited 300/677
Audited 450/677
Audited 600/677
[PASS] Dataset audit {"images": {"train": 477, "val": 105, "test": 95}, "boxes": {"train": 10050, "val": 1938, "test": 2202}, "exact_cross_split": null, "shared_blocks": []}


## 3 — COCO → YOLO without copying source images

In [4]:
from pathlib import PurePosixPath
for split in ('train','val','test'):
    (YOLO_DATA/'images'/split).mkdir(parents=True,exist_ok=True)
    (YOLO_DATA/'labels'/split).mkdir(parents=True,exist_ok=True)
mode_counter=Counter()
for name,rec in sorted(records.items()):
    split=rec['split'];src=IMAGE_PATHS[name];dest=YOLO_DATA/'images'/split/name
    if not dest.exists():
        try:dest.symlink_to(src.resolve());mode_counter['symlink']+=1
        except (OSError,NotImplementedError):shutil.copy2(src,dest);mode_counter['copy']+=1
    if not dest.is_file():raise RuntimeError(f'Failed to stage image: {dest}')
    w,h=rec['width'],rec['height']; lines=[]
    for x1,y1,x2,y2,cls in boxes_by_name[name]:
        xc=(x1+x2)/2/w;yc=(y1+y2)/2/h;bw=(x2-x1)/w;bh=(y2-y1)/h
        vals=[xc,yc,bw,bh]
        if not all(np.isfinite(vals)) or not all(0<=float(v)<=1+1e-6 for v in vals) or bw<=0 or bh<=0:
            raise ValueError(f'Invalid normalized box: {name}')
        lines.append(f'{int(cls)} '+ ' '.join(f'{float(v):.8f}' for v in vals))
    label=YOLO_DATA/'labels'/split/(Path(name).stem+'.txt')
    label.write_text('\n'.join(lines)+('\n' if lines else ''),encoding='utf-8')
import yaml
DATA_YAML=YOLO_DATA/'data.yaml'
# path as a Python str is portable inside Kaggle session
DATA_YAML.write_text(yaml.safe_dump({'path':str(YOLO_DATA),'train':'images/train','val':'images/val','test':'images/test',
                                    'nc':6,'names':{i:s for i,s in enumerate(CLASSES)}},sort_keys=False),encoding='utf-8')
for split,n in EXPECTED.items():
    images=list((YOLO_DATA/'images'/split).iterdir());labels=list((YOLO_DATA/'labels'/split).glob('*.txt'))
    assert len(images)==len(labels)==n,(split,len(images),len(labels))
    assert sum(len(p.read_text().splitlines()) for p in labels)==EXPECTED_BOX[split]
print('[PASS] Conversion',dict(mode_counter),'yaml',DATA_YAML)

[PASS] Conversion {'symlink': 677} yaml /kaggle/working/fruitcount_day33_v4/yolo_dataset/data.yaml


## 4 — Small visual QA (saved, never shown inline)

In [5]:
counts=Counter()
for c in class_counts.values():counts.update(c)
fig,axes=plt.subplots(1,2,figsize=(12,4),dpi=110)
axes[0].barh(CLASSES,[counts[c] for c in CLASSES]);axes[0].invert_yaxis();axes[0].set_title('Detection instance distribution');axes[0].set_xlabel('COCO boxes')
axes[1].bar(list(EXPECTED),list(EXPECTED.values()),label='Images')
axes[1].set_title('Official image split');axes[1].set_ylabel('Images');fig.tight_layout()
fig.savefig(PLOTS/'dataset_distribution.png');plt.close(fig)

def draw_ground_truth(name,out_path):
    im=Image.open(IMAGE_PATHS[name]).convert('RGB');im.thumbnail((900,900))
    rw=im.width/records[name]['width'];rh=im.height/records[name]['height'];d=ImageDraw.Draw(im)
    for x1,y1,x2,y2,cls in boxes_by_name[name]:
        d.rectangle((x1*rw,y1*rh,x2*rw,y2*rh),outline='#13e3a7',width=2)
    im.save(out_path);im.close()
for split in ('train','val'):
    name=sorted(n for n,r in records.items() if r['split']==split)[0]
    draw_ground_truth(name,EXAMPLES/f'{split}_ground_truth.jpg')
print('[PASS] Real GT figures saved')

[PASS] Real GT figures saved


## 5 — Train, select `best.pt` using validation, and store it locally

This is the only model-heavy phase. YOLO training itself evaluates validation per epoch; we do **not** repeat 105-image validation, run 95-image test inference, or sweep thresholds here. The next code cell checkpoints training outputs **before** optional postprocessing.

In [6]:
BEST=MODELS/'best.pt'
TRAIN_MANIFEST=REPORTS/'trained_model_manifest.json'
started=time.monotonic()

if TRAIN_MODE=='reuse':
    assert RESTORE_CHECKPOINT, 'For reuse, configure RESTORE_CHECKPOINT'
    src=Path(RESTORE_CHECKPOINT)
    assert src.is_file(),f'Checkpoint not found: {src}'
    shutil.copy2(src,BEST)
    run_dir=None
    print('[REUSE] Checkpoint copied, no training invoked')
else:
    print('[TRAIN] Starting pretrained YOLO11n training')
    model=YOLO(PRETRAINED_WEIGHTS)
    result=model.train(
        data=str(DATA_YAML),imgsz=IMG_SIZE,epochs=EPOCHS,patience=PATIENCE,
        batch=BATCH_SIZE,device=DEVICE,workers=WORKERS,seed=SEED,deterministic=True,
        amp=True,cache=False,pretrained=True,plots=True,project=str(RUNS),
        name=RUN_NAME,exist_ok=False,optimizer='auto',close_mosaic=10,
        save=True,verbose=True)
    run_dir=Path(model.trainer.save_dir)
    src=run_dir/'weights'/'best.pt'
    assert src.is_file(),f'Checkpoint was not produced: {src}'
    shutil.copy2(src,BEST)
    del result,model
    gc.collect()
    torch.cuda.empty_cache()

assert BEST.is_file() and BEST.stat().st_size>100_000
check=YOLO(str(BEST),task='detect')
actual=[check.names[i] for i in range(len(check.names))]
assert actual==CLASSES,(actual,CLASSES)
del check; gc.collect()
sha=hashlib.file_digest(BEST.open('rb'),'sha256').hexdigest() if hasattr(hashlib,'file_digest') else hashlib.sha256(BEST.read_bytes()).hexdigest()
manifest={'architecture':'yolo11n','classes':CLASSES,'image_size':IMG_SIZE,'seed':SEED,
          'checkpoint_sha256':sha,'checkpoint_size_bytes':BEST.stat().st_size,
          'source_checkpoint':str(src),'validation_selected':True,
          'selection_rule':'Ultralytics best.pt based on validation fitness',
          'training_or_reuse':TRAIN_MODE,'training_section_wall_seconds':round(time.monotonic()-started,2),
          'test_status':'not_evaluated_in_this_notebook; previously viewed V2/V3 split not blind'}
TRAIN_MANIFEST.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('[PASS] Model checkpoint saved:',BEST,'SHA-256',sha)

[TRAIN] Starting pretrained YOLO11n training
Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/fruitcount_day33_v4/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_sc

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/60      1.75G      1.096      1.332      1.034         60        960: 100% ━━━━━━━━━━━━ 120/120 1.0s/it 2:01
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 1.8s/it 25.4s
                   all        105       1937      0.511      0.367      0.355      0.268

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/60      1.75G      1.053      1.194      1.014         49        960: 100% ━━━━━━━━━━━━ 120/120 1.0s/it 2:01
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 1.8s/it 24.8s
                   all        105       1937      0.336      0.458      0.368      0.281

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/60      1.75G      1.058      1.156      1.013         30        960: 100% ━━━━━━━━━━━━ 120/120 1.0it/s 1:59
                 Class     Images  Instances      Box(

## 6 — Immediate artifact export; do not run any extra model inference

This stage writes two ZIPs **immediately after training**: first the checkpoint-only recovery archive, then full training plots/reports. It uses no per-image `predict` calls, and ZIP verification samples the compact archive rather than re-evaluating images. Output ZIPs are stored in `/kaggle/working`.

In [7]:
import contextlib
from zipfile import ZipFile, ZIP_DEFLATED

RECOVERY_ZIP=Path('/kaggle/working/FruitCount_Day33_V4_Recovery.zip')
FULL_ZIP=Path('/kaggle/working/FruitCount_Day33_V4_Training_Results.zip')

# FIRST: independent recovery artifact, produced as soon as model is ready.
with ZipFile(RECOVERY_ZIP,'w',ZIP_DEFLATED,compresslevel=1,allowZip64=True) as z:
    z.write(BEST,'models/best.pt')
    z.write(TRAIN_MANIFEST,'reports/trained_model_manifest.json')
    z.write(REPORTS/'experiment_config.json','reports/experiment_config.json')
with ZipFile(RECOVERY_ZIP) as z:
    assert z.testzip() is None
    assert {'models/best.pt','reports/trained_model_manifest.json'}<=set(z.namelist())
print('[PASS] Recovery ZIP',RECOVERY_ZIP,'bytes',RECOVERY_ZIP.stat().st_size,flush=True)

# THEN: add dataset provenance and genuine source training plots/history only.
for file in (SOURCE_README,SOURCE_LICENSE):
    out=REPORTS/('source_'+file.name)
    if not out.exists():shutil.copy2(file,out)
if run_dir is not None and run_dir.is_dir():
    for p in run_dir.iterdir():
        if not p.is_file() or p.suffix.lower() not in {'.csv','.yaml','.png','.jpg'}:continue
        if p.name.startswith(('train_batch','val_batch')):continue
        out=(PLOTS if p.suffix.lower() in {'.png','.jpg'} else REPORTS)/('train_'+p.name)
        if not out.exists():shutil.copy2(p,out)

README=REPORTS/'README.md'
README.write_text('# FruitCount V4 — Training and Recovery\n\nCheckpoint selected on validation. Training plots/history are included when available.\nNo post-training confidence tuning, count metrics or test evaluation in this notebook.\nPrevious V2/V3 test examination makes the official test development-aware, not blind.\n',encoding='utf-8')

manifest_files=[p for root in (REPORTS,PLOTS,EXAMPLES,MODELS) for p in root.rglob('*') if p.is_file() and p.name!='artifact_manifest.json']
entries=[]
for p in sorted(manifest_files):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''):h.update(chunk)
    entries.append({'path':p.relative_to(WORK).as_posix(),'size':p.stat().st_size,'sha256':h.hexdigest()})
(REPORTS/'artifact_manifest.json').write_text(json.dumps({'files':entries,'created_utc':datetime.now(timezone.utc).isoformat()},indent=2),encoding='utf-8')

with ZipFile(FULL_ZIP,'w',ZIP_DEFLATED,compresslevel=1,allowZip64=True) as z:
    for root in (REPORTS,PLOTS,EXAMPLES,MODELS):
        for p in sorted(root.rglob('*')):
            if p.is_file():z.write(p,p.relative_to(WORK).as_posix())
with ZipFile(FULL_ZIP) as z:
    assert z.testzip() is None
    assert 'models/best.pt' in z.namelist()
    assert 'reports/trained_model_manifest.json' in z.namelist()
print('[PASS] Full training results:',FULL_ZIP, 'MiB',round(FULL_ZIP.stat().st_size/1048576,2),flush=True)
print('[DONE] Save Version training/export completed. No extra model inference follows.',flush=True)

[PASS] Recovery ZIP /kaggle/working/FruitCount_Day33_V4_Recovery.zip bytes 4984937
[PASS] Full training results: /kaggle/working/FruitCount_Day33_V4_Training_Results.zip MiB 6.51
[DONE] Save Version training/export completed. No extra model inference follows.


## Completion contract

**Required outputs**: `FruitCount_Day33_V4_Recovery.zip` and `FruitCount_Day33_V4_Training_Results.zip`. Both include the real `models/best.pt` and its checksum manifest. Training plots/history are included in the latter when the trainer produced them.

**Not included (intentionally):** duplicate validation, confidence tuning, raw/count MAE/RMSE, or development-aware test detection. Those belong in a **separate evaluation notebook**, ideally using the already trained checkpoint as a Kaggle Input and with resource/time constraints. Do not claim an unmeasured result.

**Existing V3 checkpoint:** if it becomes accessible, prefer to use it rather than retrain. Set `TRAIN_MODE='reuse'` and `RESTORE_CHECKPOINT` to the actual mounted checkpoint path in this notebook to package it without GPU training. The version currently running on Kaggle is independent of this uploaded notebook.